# 02 — Sinh trajectories (teacher Qwen3-14B, rejection sampling trên TRAIN)

Chỉ dùng train + dữ liệu tự sinh (luật thi). Public test chỉ đọc `questions.jsonl` để kiểm tra trùng lặp. Kết quả lưu vào `DRIVE_ROOT/training/data`. Chạy lại cell cuối để tiếp tục khi bị ngắt (resumable).

In [ ]:
# === Setup (chạy mỗi lần mở runtime) ===
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/DataMate_ZAC2026'   # thư mục Drive chung (Add shortcut to My Drive nếu được share)
REPO_BRANCH = 'feat/slm-agent'
import os, subprocess
if not os.path.exists('/content/AI-Challenge'):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, 'https://github.com/VTin206/AI-Challenge.git', '/content/AI-Challenge'], check=True)
%cd /content/AI-Challenge
!git pull -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SPLITS = ['train', 'public_test']
# === Giải nén dữ liệu từ Drive vào /content/data (đĩa local, nhanh hơn Drive) ===
import zipfile, os
os.makedirs('/content/data', exist_ok=True)
for split in SPLITS:
    if not os.path.exists(f'/content/data/{split}'):
        zipfile.ZipFile(f'{DRIVE_ROOT}/data/{split}/{split}.zip').extractall('/content/data')
!ls /content/data

In [ ]:
!pip install -q -e . vllm

In [ ]:
!python training_code/data_generation/split_holdout.py --split /content/data/train --cache-dir /content/cache --out configs/split.json
!cp configs/split.json {DRIVE_ROOT}/training/split.json 2>/dev/null || (mkdir -p {DRIVE_ROOT}/training && cp configs/split.json {DRIVE_ROOT}/training/split.json)

In [ ]:
!python training_code/data_generation/gen_trajectories.py --split /content/data/train --public-test /content/data/public_test \
    --split-file configs/split.json --out {DRIVE_ROOT}/training/data --work /content/work --max-hours 5 --attempts 1 --retry-attempts 2

In [ ]:
!python training_code/src/manifest.py --data {DRIVE_ROOT}/training/data --zips {DRIVE_ROOT}/data/train/train.zip --out {DRIVE_ROOT}/training/data_manifest.json
!cat {DRIVE_ROOT}/training/data/yield_report.json